In [1]:
"""
update_allergen_summary.py
==========================
Tổng hợp allergen_tags từ ingredients → cập nhật allergen_summary trong dishes
"""

import sqlite3
import json
from pathlib import Path

DB_PATH = "../cookpad_clean.db"  # <-- sửa đường dẫn

conn = sqlite3.connect(DB_PATH)

# Fetch tất cả dish_ingredient × allergen_tags
rows = conn.execute("""
    SELECT
        di.recipe_id,
        i.allergen_tags
    FROM dish_ingredient di
    JOIN ingredients i ON i.id = di.ingredient_id
    WHERE di.quantity_g > 0
      AND i.allergen_tags IS NOT NULL
      AND i.allergen_tags != ''
      AND i.allergen_tags != '[]'
""").fetchall()

print(f"Fetched {len(rows)} ingredient rows có allergen_tags")

# Gom allergens theo recipe_id
recipe_allergens: dict[str, set] = {}
for recipe_id, allergen_tags in rows:
    allergens = set()
    tag = allergen_tags.strip()
    try:
        if tag.startswith("["):
            parsed = json.loads(tag)
        else:
            parsed = [t.strip() for t in tag.split(",") if t.strip()]
        allergens.update(parsed)
    except (json.JSONDecodeError, AttributeError):
        pass

    if allergens:
        recipe_allergens.setdefault(recipe_id, set()).update(allergens)

print(f"Recipes có allergen: {len(recipe_allergens)}")

# Cập nhật dishes.allergen_summary
updated = 0
cur = conn.cursor()

for recipe_id, allergens in recipe_allergens.items():
    allergen_json = json.dumps(sorted(allergens), ensure_ascii=False)
    cur.execute("""
        UPDATE dishes
        SET allergen_summary = ?
        WHERE id = ?
    """, (allergen_json, recipe_id))
    updated += cur.rowcount

# Dishes không có allergen → set []
cur.execute("""
    UPDATE dishes
    SET allergen_summary = '[]'
    WHERE allergen_summary IS NULL
       OR allergen_summary = ''
""")
no_allergen = cur.rowcount

conn.commit()
conn.close()

print(f"\n✅ Updated allergen_summary : {updated} dishes")
print(f"✅ Set '[]' cho             : {no_allergen} dishes không có allergen")

# Verify
conn = sqlite3.connect(DB_PATH)
row = conn.execute("""
    SELECT
        COUNT(*) as total,
        SUM(CASE WHEN allergen_summary = '[]' THEN 1 ELSE 0 END) as no_allergen,
        SUM(CASE WHEN allergen_summary != '[]' THEN 1 ELSE 0 END) as has_allergen
    FROM dishes
""").fetchone()
conn.close()

print(f"\n=== Verify ===")
print(f"  Total dishes    : {row[0]}")
print(f"  Có allergen     : {row[2]}")
print(f"  Không allergen  : {row[1]}")

Fetched 5109 ingredient rows có allergen_tags
Recipes có allergen: 3910

✅ Updated allergen_summary : 3910 dishes
✅ Set '[]' cho             : 0 dishes không có allergen

=== Verify ===
  Total dishes    : 8282
  Có allergen     : 4478
  Không allergen  : 3804


In [4]:
"""
export_vegan_few_ingredients.py
================================
Xuất tất cả món is_vegan=1 có số nguyên liệu < 3
ra file JSON để review và fix thủ công
"""

import sqlite3
import json
from pathlib import Path

DB_PATH     = "../cookpad_clean.db"   # <-- sửa đường dẫn
OUTPUT_FILE = "./vegan_few_ingredients_3.json"

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

# Lấy danh sách món is_vegan=1 có < 3 nguyên liệu
dishes = conn.execute("""
    SELECT 
        d.id,
        d.title,
        COUNT(di.ingredient_id) as so_nguyen_lieu
    FROM dishes d
    LEFT JOIN dish_ingredient di ON di.recipe_id = d.id
    WHERE d.is_vegan !=-1
    GROUP BY d.id, d.title
    HAVING COUNT(di.ingredient_id) < 3
    ORDER BY so_nguyen_lieu ASC, d.title
""").fetchall()

print(f"Tìm thấy {len(dishes)} món is_vegan=1 có < 3 nguyên liệu")

# Với mỗi món, lấy chi tiết nguyên liệu
result = []
for dish in dishes:
    ingredients = conn.execute("""
        SELECT 
            di.ingredient_id,
            i.name,
            i.category,
            i.is_animal_based,
            di.quantity_g
        FROM dish_ingredient di
        LEFT JOIN ingredients i ON i.id = di.ingredient_id
        WHERE di.recipe_id = ?
    """, (dish["id"],)).fetchall()

    result.append({
        "id":               dish["id"],
        "title":            dish["title"],
        "so_nguyen_lieu":   dish["so_nguyen_lieu"],
        "ingredients": [
            {
                "ingredient_id":  ing["ingredient_id"],
                "name":           ing["name"],
                "category":       ing["category"],
                "is_animal_based": ing["is_animal_based"],
                "quantity_g":     ing["quantity_g"],
            }
            for ing in ingredients
        ]
    })

conn.close()

# Ghi ra JSON
output_path = Path(OUTPUT_FILE)
output_path.write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(f"✅ Đã xuất {len(result)} món → {output_path.resolve()}")

# Thống kê nhanh


print(f"\n=== Phân phối ===")


Tìm thấy 2427 món is_vegan=1 có < 3 nguyên liệu
✅ Đã xuất 2427 món → D:\dream_project\daily_mate_implement\db\update_sql\vegan_few_ingredients_3.json

=== Phân phối ===
